## Linking a dataset of real historical persons with Deterrministic Rules

While Splink is primarily a tool for probabilistic records linkage, it includes functionality to perform deterministic (i.e. rules based) linkage.

Significant work has gone into optimising the performance of rules based matching, so Splink is likely to be significantly faster than writing the basic SQL by hand.

In this example, we deduplicate a 50k row dataset based on historical persons scraped from wikidata. Duplicate records are introduced with a variety of errors introduced. The probabilistic dedupe of the same dataset can be found at `Deduplicate 50k rows historical persons`.


In [ ]:
# Uncomment and run this cell if you're running in Google Colab.
# !pip install splink

In [1]:
import pandas as pd

from splink import splink_datasets

pd.options.display.max_rows = 1000
df = splink_datasets.historical_50k
df.head()

,unique_id,cluster,full_name,first_and_surname,first_name,surname,dob,birth_place,postcode_fake,gender,occupation
0,Q2296770-1,Q2296770,"thomas clifford, 1st baron clifford of chudleigh",thomas chudleigh,thomas,chudleigh,1630-08-01,devon,tq13 8df,male,politician
1,Q2296770-2,Q2296770,thomas of chudleigh,thomas chudleigh,thomas,chudleigh,1630-08-01,devon,tq13 8df,male,politician
2,Q2296770-3,Q2296770,tom 1st baron clifford of chudleigh,tom chudleigh,tom,chudleigh,1630-08-01,devon,tq13 8df,male,politician
3,Q2296770-4,Q2296770,thomas 1st chudleigh,thomas chudleigh,thomas,chudleigh,1630-08-01,devon,tq13 8hu,None,politician
4,Q2296770-5,Q2296770,"thomas clifford, 1st baron chudleigh",thomas chudleigh,thomas,chudleigh,1630-08-01,devon,tq13 8df,None,politician


When defining the settings object, specity your deterministic rules in the `blocking_rules_to_generate_predictions` key.

For a deterministic linkage, the linkage methodology is based solely on these rules, so there is no need to define `comparisons` nor any other parameters required for model training in a probabilistic model.


Prior to running the linkage, it's usually a good idea to check how many record comparisons will be generated by your deterministic rules:


In [2]:
from splink import DuckDBAPI, block_on
from splink.blocking_analysis import (
    cumulative_comparisons_to_be_scored_from_blocking_rules_chart,
)

db_api = DuckDBAPI()
cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables=df,
    blocking_rules=[
        block_on("first_name", "surname", "dob"),
        block_on("surname", "dob", "postcode_fake"),
        block_on("first_name", "dob", "occupation"),
    ],
    db_api=db_api,
    link_type="dedupe_only",
)

alt.Chart(...)

In [3]:
from splink import Linker, SettingsCreator

settings = SettingsCreator(
    link_type="dedupe_only",
    blocking_rules_to_generate_predictions=[
        block_on("first_name", "surname", "dob"),
        block_on("surname", "dob", "postcode_fake"),
        block_on("first_name", "dob", "occupation"),
    ],
    retain_intermediate_calculation_columns=True,
)

linker = Linker(df, settings, db_api=db_api)


The results of the linkage can be viewed with the `deterministic_link` function.


In [4]:
df_predict = linker.inference.deterministic_link()
df_predict.as_pandas_dataframe().head()

,unique_id_l,unique_id_r,first_name_l,first_name_r,surname_l,surname_r,postcode_fake_l,postcode_fake_r,occupation_l,occupation_r,dob_l,dob_r,match_key
0,Q2296770-1,Q2296770-3,thomas,tom,chudleigh,chudleigh,tq13 8df,tq13 8df,politician,politician,1630-08-01,1630-08-01,1
1,Q2296770-2,Q2296770-3,thomas,tom,chudleigh,chudleigh,tq13 8df,tq13 8df,politician,politician,1630-08-01,1630-08-01,1
2,Q2296770-3,Q2296770-5,tom,thomas,chudleigh,chudleigh,tq13 8df,tq13 8df,politician,politician,1630-08-01,1630-08-01,1
3,Q2296770-4,Q2296770-5,thomas,thomas,chudleigh,chudleigh,tq13 8hu,tq13 8df,politician,politician,1630-08-01,1630-08-01,0
4,Q2296770-5,Q2296770-7,thomas,tom,chudleigh,chudleigh,tq13 8df,tq13 8df,politician,None,1630-08-01,1630-08-01,1


Which can be used to generate clusters.

Note, for deterministic linkage, each comparison has been assigned a match probability of 1, so to generate clusters, set `threshold_match_probability=1` in the `cluster_pairwise_predictions_at_threshold` function.


In [5]:
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    df_predict
)

Completed iteration 1, num representatives needing updating: 94
Completed iteration 2, num representatives needing updating: 10
Completed iteration 3, num representatives needing updating: 0


In [6]:
clusters.as_pandas_dataframe(limit=5)

,cluster_id,unique_id,cluster,full_name,first_and_surname,first_name,surname,dob,birth_place,postcode_fake,gender,occupation
0,Q2296770-1,Q2296770-1,Q2296770,"thomas clifford, 1st baron clifford of chudleigh",thomas chudleigh,thomas,chudleigh,1630-08-01,devon,tq13 8df,male,politician
1,Q2296770-1,Q2296770-2,Q2296770,thomas of chudleigh,thomas chudleigh,thomas,chudleigh,1630-08-01,devon,tq13 8df,male,politician
2,Q2296770-1,Q2296770-3,Q2296770,tom 1st baron clifford of chudleigh,tom chudleigh,tom,chudleigh,1630-08-01,devon,tq13 8df,male,politician
3,Q2296770-1,Q2296770-4,Q2296770,thomas 1st chudleigh,thomas chudleigh,thomas,chudleigh,1630-08-01,devon,tq13 8hu,None,politician
4,Q2296770-1,Q2296770-5,Q2296770,"thomas clifford, 1st baron chudleigh",thomas chudleigh,thomas,chudleigh,1630-08-01,devon,tq13 8df,None,politician


These results can then be passed into the `Cluster Studio Dashboard`.


In [ ]:
linker.visualisations.cluster_studio_dashboard(
    df_predict,
    clusters,
    "../../results/50k_deterministic_cluster.html",
    sampling_method="by_cluster_size",
    overwrite=True,
)

from IPython.display import IFrame

# IFrame(src="../../results/50k_deterministic_cluster.html", width="100%", height=1200)

# Open the file in your browser
import webbrowser
webbrowser.open("../../results/50k_deterministic_cluster.html")

True

gio: file:///home/enginux/stage/splink/results/50k_deterministic_cluster.html: Failed to find default application for content type ‘text/html’
